# FieldAE check (`ae_check.ipynb`)

AE only: where the true 48³ field and the AE-decoded field agree and disagree, and how the occupancy threshold changes that. Trains nothing and changes no existing output.

1. **Threshold scan**: reads `eval/thr_<t>/` (ae-eval outputs) and plots every acceptance metric against the threshold.
2. **Per event**: one row per val event: true photons (sum over y), decoded photons, map of extra / missed cells, longitudinal profile, radial profile. By default the events are picked from the scan: smallest, median, largest, worst long_l1, worst q_total.
3. **Pooled**: `N_POOL` val events together: p_occ split by truth, photons per cell class, per-cell photon accuracy on matched cells, and each event's total-photon ratio against event size.

- `THR` defaults to `field.occ_threshold` of CONFIG, or set it with `--set THR=0.9`. `COMPARE_THR` is drawn dotted for comparison.
- AE decodes share the cache `<case>/viz/cache/` with `viz.ipynb` (per checkpoint step and event). Threshold-dependent quantities are recomputed from the decoded field, so changing the threshold does not re-run the AE.
- Figures go to `<case>/viz/figs_ae/`; an existing name gets a timestamp instead of being overwritten.
- Headless: `python scripts/run_notebook.py notebooks/ae_check.ipynb --set THR=0.9`, or `jobs/notebook_cpu.sub`.


In [ ]:
# parameters (run_notebook.py --set NAME=VALUE replaces these lines)
CONFIG = 'configs/v2.yaml'
CASE = 'v2a'
DEVICE = 'cpu'             # 'cuda' only speeds up the first (uncached) AE decode
AE_CKPT = 'best'
THR = None                 # None: field.occ_threshold of CONFIG
COMPARE_THR = 0.5          # drawn dotted for comparison (None: skip)
EVENTS = None              # None: picked from eval/thr_<THR>/per_event.jsonl, else first val events
SCAN_PREFIX = 'thr_'       # ae-eval outputs eval/<SCAN_PREFIX><t>/
N_POOL = 32                # val events in the pooled figure
FIG_SUBDIR = 'figs_ae'

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / 'sparseshower').is_dir() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
import numpy as np
import matplotlib.pyplot as plt
from sparseshower.common import config
from sparseshower.loader import case_dir, meta_for
from sparseshower import viz_ae
c = config(CONFIG)
CASE_DIR = case_dir(c, CASE)
THR = float(c['field']['occ_threshold'] if THR is None else THR)
FIG_DIR = CASE_DIR / 'viz' / FIG_SUBDIR
def save(fig, name):
    if fig is not None:
        print('saved', viz_ae.save_new(fig, FIG_DIR, name))
        plt.show()
print('project  :', ROOT)
print('config   :', CONFIG, '| data', c['paths']['processed'])
print('case dir :', CASE_DIR, '(exists)' if CASE_DIR.exists() else '(MISSING)')
print('threshold:', THR, '| compare', COMPARE_THR)
print('figures  ->', FIG_DIR)
if c.get('synthetic'):
    print('*** SYNTHETIC data (smoke config): nothing here is CORSIKA ***')
HAVE_AE = (CASE_DIR / f'ae_{AE_CKPT}.pt').exists()
if not HAVE_AE:
    print(f'no ae_{AE_CKPT}.pt yet: sections 2 and 3 are skipped')

## 1. Threshold scan

Left: number of non-empty 48³ cells (predicted / true), and extra and missed cells (both divided by the true non-empty count); the grey band is the n48 threshold. Right: lost photons, longitudinal L1, radial L1, total-photon error; dotted lines are their thresholds. The grey vertical dashed line is the current `THR`. "Missed cells" is derived from per_event: extra cells − (ratio − 1).


In [ ]:
scan = viz_ae.load_scan(c, CASE, SCAN_PREFIX)
if scan:
    save(viz_ae.plot_scan(c, scan, THR), f'1_scan_{AE_CKPT}')
    print(f"{'thr':>6} {'n48 ratio':>10} {'lost':>9} {'long_l1':>8} {'rad_l1':>8} {'q_total':>8}")
    for t, s, pe in scan:
        m = s['metrics']
        ratio = np.mean([r['n48_pred'] / max(r['n48_true'], 1) for r in pe])
        print(f"{t:>6} {ratio:>10.4f} {m['lost_photon_frac']['mean']:>9.5f} {m['field_long_l1']['mean']:>8.4f} "
              f"{m['field_rad_l1']['mean']:>8.4f} {m['q_total_rel_err']['mean']:>8.4f}")
else:
    print(f'no eval/{SCAN_PREFIX}<t>/ outputs (run ae-eval at several thresholds first)')

## 2. Per event

Column 3 colours each y-column of cells: grey means everything agrees, red means the column has extra cells (predicted non-empty, truth empty), blue means missed cells (truth non-empty, predicted empty), purple both. In the profile plots black is truth, red dashed the decode at `THR`, blue dotted the decode at `COMPARE_THR`.


In [ ]:
if HAVE_AE:
    if EVENTS is None:
        picks = viz_ae.pick_events(c, CASE, f'{SCAN_PREFIX}{THR:g}')
    else:
        picks = [(int(e), 'chosen') for e in EVENTS]
    labels = dict(picks)
    rows, stats, pidx = viz_ae.load_rows(c, CASE, [e for e, _ in picks], AE_CKPT, DEVICE)
    save(viz_ae.plot_events(c, rows, labels, THR, stats, pidx, COMPARE_THR, AE_CKPT), f'2_events_thr{THR:g}_{AE_CKPT}')
    print(f"{'event':>6} {'why':<40} {'n48':>6} {'ratio':>6} {'extra':>6} {'missed':>6} {'lost':>7} {'Q ratio':>7} {'long':>7} {'rad':>7}")
    for r in rows:
        m = viz_ae.at_threshold(c, r, THR, stats, pidx)['metrics']
        e = int(r['eid'])
        print(f"{e:>6} {labels[e]:<40} {m['n48_true']:>6} {m['n48_ratio']:>6.3f} {m['extra']:>6.3f} {m['missed']:>6.3f} "
              f"{m['lost']:>7.4f} {m['q_ratio']:>7.4f} {m['long_l1']:>7.4f} {m['rad_l1']:>7.4f}")

## 3. Pooled

- **Panel 1**: p_occ on truly non-empty cells and on truly empty cells (empty cells only where p > 0.001). The two vertical lines are the two thresholds: black mass left of a line is missed cells, red mass right of it is extra cells.
- **Panel 2**: photons per cell for matched, missed and extra cells. The legend gives each class's cell count and total photons.
- **Panel 3**: true vs decoded photons per matched cell; the red line is the diagonal. The title gives the mean (bias) and standard deviation (spread) of the log10 ratio.
- **Panel 4**: decoded / true total photons per event against event size (true non-empty cells); the red band is ±1%.


In [ ]:
if HAVE_AE:
    pool_ids = [int(e) for e in meta_for(c)['split']['val']][:N_POOL]
    rows_pool, stats, pidx = viz_ae.load_rows(c, CASE, pool_ids, AE_CKPT, DEVICE)
    save(viz_ae.plot_pooled(c, rows_pool, THR, stats, pidx, COMPARE_THR), f'3_pooled_thr{THR:g}_{AE_CKPT}')
    _, per_event = viz_ae.pooled(c, rows_pool, THR, stats, pidx)
    for k in ('n48_ratio', 'extra', 'missed', 'lost', 'q_ratio', 'long_l1', 'rad_l1'):
        v = np.array([e[k] for e in per_event])
        print(f'{k:<10} mean {v.mean():.4f}  p10 {np.percentile(v, 10):.4f}  p90 {np.percentile(v, 90):.4f}')